# 🔴 Hard: Stable Diffusion Foundations — Toy DDPM

Train a complete two-dimensional diffusion model: add noise to target points, teach a time-conditioned MLP to predict that noise, then turn Gaussian noise into new samples with the reverse DDPM chain.

> **Shape legend:** B = batch size, D = point dimension (2), and T = number of diffusion steps (100).


## 1. Forward diffusion

Choose a variance schedule $\beta_1,\ldots,\beta_T$, and define

$$\alpha_t = 1 - \beta_t, \qquad \bar\alpha_t = \prod_{s=1}^{t}\alpha_s.$$

Rather than applying every earlier noise step, sample a noisy point at timestep $t$ directly:

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1 - \bar\alpha_t}\,\epsilon, \qquad \epsilon \sim \mathcal N(0, I).$$

In code, x0 and noise have shape (B, D). Selecting the per-example schedule values as (B, 1) lets PyTorch broadcast them over the D coordinates.

## 2. Your task

Implement both functions below.

1. **diffusion_loss**: choose one random integer timestep per item, construct $x_t$, normalize time to $[0, 1]$, and return the noise-prediction MSE.
2. **sample_ddpm**: start at $x_T$, loop from $T - 1$ to $0$, predict noise, estimate $\hat{x}_0$, and sample the DDPM posterior. At step $0$, return the posterior mean without adding random noise.

$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar\alpha_t}\,\epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}.$$

For $t > 0$:

$$\mu_t = \frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1 - \bar\alpha_t}\hat{x}_0 + \frac{\sqrt{\alpha_t}(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t}x_t, \qquad \sigma_t^2 = \frac{\beta_t(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t}.$$


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim


def sample_p1(n, device=None):
    """Return target samples with shape (n, 2)."""
    angle = torch.rand(n, device=device) * 2 * torch.pi  # (n,)
    radius = 2 + 0.1 * torch.randn(n, device=device)    # (n,)
    return torch.stack(
        (radius * torch.cos(angle), radius * torch.sin(angle) * torch.cos(angle)),
        dim=1,
    )  # (n, 2)


def make_noise_schedule(n_steps, device):
    """Return betas, alphas, and alpha_bars; each has shape (T,)."""
    betas = torch.linspace(1e-4, 2e-2, n_steps, device=device)  # (T,)
    alphas = 1 - betas                                           # (T,)
    alpha_bars = torch.cumprod(alphas, dim=0)                    # (T,)
    return betas, alphas, alpha_bars


class NoisePredictor(nn.Module):
    """Predict 2D noise from a noisy 2D point and normalized time."""

    def __init__(self, hidden_dim=128):
        super().__init__()
        # Input is [x_t[:, 0], x_t[:, 1], normalized_t]: (B, 3)
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2),  # predicted epsilon: (B, 2)
        )

    def forward(self, x, t):
        # x: (B, 2); t: (B,) or (B, 1)
        if t.ndim == 1:
            t = t[:, None]
        return self.net(torch.cat((x, t), dim=1))  # (B, 2)


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE
def diffusion_loss(model, x0, alpha_bars):
    """Return MSE(predicted_noise, sampled_noise).

    x0 has shape (B, D), alpha_bars has shape (T,).
    Hint: index alpha_bars with a (B,) timestep tensor, then add a
    trailing dimension to get (B, 1) for broadcasting over D.
    """
    # 1. Sample t: (B,); select alpha_bar_t: (B, 1).
    # 2. Sample noise and construct xt: both (B, D).
    # 3. Normalize t to (B, 1), predict noise, and return MSE.
    pass


@torch.no_grad()
def sample_ddpm(model, x_T, betas, alphas, alpha_bars):
    """Reverse x_T of shape (B, D) into generated samples of shape (B, D).

    Loop backward over step. The schedule values at one step are scalars;
    they naturally broadcast over the batch and coordinate dimensions.
    """
    # Hint: use alpha_bars[step - 1] when step > 0; for step == 0,
    # alpha_bar_prev should be 1. Restore model.train(...) before returning.
    pass


## 3. Train and compare

Each update uses a new target batch of shape (512, 2). Once both functions are implemented, run the cells below: the generated point cloud should gradually resemble the target distribution.

Real Stable Diffusion uses the same diffusion objective in VAE latent space, replaces this MLP with a U-Net, and conditions the denoiser on text embeddings.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_steps = 100

betas, alphas, alpha_bars = make_noise_schedule(n_steps, device)
model = NoisePredictor().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for step in range(1, 4_001):
    x0 = sample_p1(512, device)  # (B=512, D=2)
    loss = diffusion_loss(model, x0, alpha_bars)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 800 == 0:
        print(f'step {step:>4d} | loss: {loss.item():.4f}')

generated = sample_ddpm(
    model, torch.randn(1_000, 2, device=device), betas, alphas, alpha_bars
).cpu()  # (1_000, 2)
target = sample_p1(1_000).cpu()  # (1_000, 2)

plt.figure(figsize=(7, 5))
plt.scatter(target[:, 0], target[:, 1], s=4, alpha=0.5, label='target')
plt.scatter(generated[:, 0], generated[:, 1], s=4, alpha=0.5, label='generated')
plt.axis('equal')
plt.legend()
plt.show()


In [ ]:
from torch_judge import check

check('stable_diffusion')
